# Lab 3: Topographic Maps

Electrodes are small sensors, placed on the scalp, sometimes with conductive gel to record the signal. Their placement on the scalp is based on the international 10-20 System, developed by Dr. Herbert Jasper in 1950s. There are other placement systems too, however, the 10-20 system is widely used. In the 10-20 system, '10' and '20' refer to the distance between electrodes, i.e. 10% or 20% of the total length (front-to-back or right-to-left). Each electrode location is labeled with character (s) followed by a number (e.g. 'F2'). The characters are named after the region of the brain, such as Prefrontal (Fp), Frontal (F), Occupital (O), Parietal (P), Temporal (T), and center (C). A pictorial diagram below shows the 10-20 system with electrodes position colored with region of the brain.

<center>
<img src="./images/10-20_System.png" alt="" width=90%/>
<br>
Figure 1: 10-20 System</center>

Plotting eletrode values as a topographic heatmap provides two important aspect of the EEG Recording. First it maps the eletrodes values to the brain regions and gives insight of the regional brain activities. Second, colors of heatmap provides a comparative analysis of the activities.

In this worksheet we will learn to produce the topographic maps of from EEG recordings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spkit as sp

# Let's import EEG Sample

In [ ]:
X, fs, ch_names = sp.data.eeg_sample_14ch()
print('Shape of EEG : ',X.shape)
print('Sampling Frequency   :',fs)
print('Electrodes   :',ch_names)

To create a topographic map, we need postion of eletrodes and their respective values. 

From above sample, we have names of the each channel, which will be used to get their postion on a 2D plane accroding to 10-20 system. Let's extract the position of 14-channels

In [ ]:
pos_2d , _ = sp.eeg.s1020_get_epos2d(ch_names)
pos_2d.shape

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(*pos_2d.T, 'o')
plt.axis('off')
plt.show()

## Topographic map with raw EEG values (amplitude)

Let's extract 14 raw values of the electrodes from EEG data at tx = 1.5s  (ntx = 1.5*fs = 192)

In [ ]:
tx = 1.5
ntx = int(tx*fs)
x = X[ntx,:]

X1 = X[:int(fs*8)]
t = np.arange(X1.shape[0])/fs
plt.figure(figsize=(10,4))
plt.plot(t,X1 +np.arange(14)*100,'C0')
plt.yticks(np.arange(14)*100, ch_names)
plt.xlim([t[0],t[-1]])
plt.axvline(tx,color='k',alpha=0.9,ls='--')
plt.grid(alpha=0.4)
plt.show()

In [ ]:
x.min(), x.max()

Raw EEG values at tx=1.5, show the lowest amplitude around -6 and maximum aplitude around 26. We can visulaise them as scatter plot first.

In [ ]:
plt.figure(figsize=(4,3.5))
plt.scatter(pos_2d[:,0],pos_2d[:,1],c=x,s=300,cmap='jet')
plt.axis('off')
plt.show()

Although, above scatter plot shows the electrodes and corresponding values as a color, it is prefered to visualise the full topographic heatmap by interpolating the values in between to fill the gaps.

There are several libraries that provide functionalities to visualise a topographic of a EEG data, We will use spkit function that is very straightforward.

In [ ]:
pos_2d

In [ ]:
Zi = sp.eeg.topomap(data=x,pos=pos_2d,ch_names=ch_names,shownames=False,
                       show=True,vmin=x.min(),vmax=x.max())

In [ ]:
tx = 1.5
ntx = int(tx*fs)
x1 = X[ntx,:]


tx = 2.5
ntx = int(tx*fs)
x2 = X[ntx,:]

x1, x2

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,4))
Z1,im1 = sp.eeg.topomap(data=x1,ch_names=ch_names,shownames=False,axes=ax[0],return_im=True)
Z2,im2 = sp.eeg.topomap(data=x2,ch_names=ch_names,shownames=False,axes=ax[1],return_im=True)

#ax[0].set_title(r'$\alpha$ : [8-14] Hz')
#ax[1].set_title(r'$\beta$ : [15-32] Hz')
plt.colorbar(im1, ax=ax[0],label='amplitude (µV)')
plt.colorbar(im2, ax=ax[1],label='amplitude (µV)')
plt.show()

<div style="padding: 5px; border: 2px dashed darkblue;">

**Tasks/Activities/Quetions**

* Q1: Plot a topographic map for given EEG Data with values at tx = 1s, 2s, 3s and 4s.
* Q2: What are the differences you can observe? Are there any limitations in intereprestation of these plots?
</div>

In [ ]:
txs = [1, 2, 3, 4]
ntx = [int(ti*fs) for ti in txs]
xs = [X[ni] for ni in ntx]
len(xs)

In [ ]:
plt.figure(figsize=(15,4))
for i in range(len(xs)):
    plt.subplot(1,4,i+1)
    Zi = sp.eeg.topomap(data=xs[i],pos=pos_2d,ch_names=ch_names,shownames=False,show=True,vmin=xs[i].min(),vmax=xs[i].max())
    plt.title(f'{txs[i]}s')
plt.show()

# Topographic map with Energy/Power

Topographic heatmap with amplitude values at a single time instance have a serveral limitation in terms of interepretation. 

* The amplitude of EEG changes quite repidly over time, and the reletive comparision of amplitude values will also change.
* A negative amplitude value also indicates the brain activity,


One of the solution is to use topographic map for values of electrodes computed for a duration of the time, instead of a single time instance. There are sveral values from EEG signal can be computed that have significant meaning associated to brain activity, such as the energy of the signal.


For this we can compute energy (or power) of the signal 1 second duration. 

Let's compute the energy of each electrode signal from 1 to 2 duration

In [ ]:
t1,t2 = int(fs*1),int(fs*2)
Xi = X[t1:t2]
Xi.shape

In [ ]:
X1 = X[:int(fs*8)]
t = np.arange(X1.shape[0])/fs
plt.figure(figsize=(10,4))
plt.plot(t,X1 +np.arange(14)*100,'C0')
plt.yticks(np.arange(14)*100, ch_names)
plt.xlim([t[0],t[-1]])
plt.axvspan(t1/fs,t2/fs, color='g',alpha=0.5)
plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Power ( = average energy)

Ei = np.mean(Xi**2,axis=0)
Ei.min(), Ei.max()

In [ ]:
Ei

In [ ]:
Zi = sp.eeg.topomap(data=Ei,pos=pos_2d,ch_names=ch_names,shownames=False,vmin=Ei.min(),vmax=Ei.max())

<div style="padding: 5px; border: 2px dashed darkblue;">

**Tasks/Activities/Quetions**

* Q3: Plot a topographic map for given EEG Data with power of duration between (2-3)s, (3-4)s, (4-5)s, and (5-6)s.
</div>

In [ ]:
txs = [2, 3, 4, 5]
Eis = [np.mean(X[int(fs*ti):int(fs*(ti+1))]**2,axis=0) for ti in txs]

In [ ]:
[(ei.min(),ei.max()) for ei in Eis]

In [ ]:
plt.figure(figsize=(12,4))
for i in range(len(xs)):
    plt.subplot(1,4,i+1)
    Zi = sp.eeg.topomap(data=Eis[i],pos=pos_2d,ch_names=ch_names,shownames=False)
    plt.title(f'{txs[i]}s')
plt.show()

<div style="padding: 5px; border: 2px dashed darkblue;">

**Tasks/Activities/Quetions**

* Q4: Consider the 9 to 16 seconds of duration from given EEG sample, (as shown below), Compute the power of each channel for 1 second duration continuesly (9-10, 10-11, 11-12, and so on) and plot them as topographic maps.
* Q5: Are these 1 second segments comparable? Is it fair to compare these 1 second segments? justify your answers.
</div>

In [ ]:
X1 = X[int(fs*9):]
t = np.arange(X1.shape[0])/fs
plt.figure(figsize=(10,5))
plt.plot(t,X1 +np.arange(14)*100,'C0')
plt.plot(t,X1*0 +np.arange(14)*100,'k',lw=0.1)
plt.yticks(np.arange(14)*100, ch_names)
plt.xlim([t[0],t[-1]])
plt.grid(alpha=0.4)
plt.show()

<div style="padding: 5px; border: 2px dashed darkblue;">

**Tasks/Activities/Quetions**

* Q6: Recall the different file structures from previous worksheet and mean values of the EEG recordings. Do these mean values affect the interpretation of topographic maps?
* Q7: Consider a segment of EEG from previous worksheet, as shown in below. Before computing Energy (or power) of each channel for given segment, can you enticipate, which eletrode (channel) will have highest value and why?
</div>

In [ ]:
import glob
files = glob.glob('./data/*.csv')
files.sort()
Dx = pd.read_csv(files[2])
col_names = list(Dx)
eeg_col = col_names[1:15]
X = Dx[eeg_col].to_numpy()
fs = 128 # from project -https://PhyAAt.github.io
X.shape

In [ ]:
t1,t2 = int(fs*40),int(fs*55)

nCh = len(eeg_col) #number of channels
sep = 100

X1 = X[t1:t2].copy()
t = np.arange(X1.shape[0])/fs
plt.figure(figsize=(10,4))
plt.plot(t,X1 +np.arange(nCh)*sep,'C0')
plt.yticks(np.arange(nCh)*sep, eeg_col)
plt.xlim([t[0],t[-1]])
plt.grid(alpha=0.4)
plt.show()

<div style="padding: 5px; border: 2px solid darkblue;">

**Conclusion/Next**

As you might have observed that mean values and some sudden spkies in the EEG signal can mislead our interepretations of topographic maps. In next worksheet, we will learn to clean EEG signal by removing noise and artifacts, which will make our interepretation much more reliable.

</div>